# Run felsenstein on simulated and recovered trees

Some steps to take

- Run felsenstein once per tree. Its possible multiple SNPs share the same tree.
we can store which tree contain a snp of interest
- chnage so that tree length is cached, we can reuse so that we dont have to recqlculate

In [159]:
import tskit
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display, SVG
from scipy.linalg import expm
import sys
sys.path.insert(0, "../python_code")
from python_code import felsenstein_codes
from felsenstein_codes import felsenstein_backwards_general
from collections import defaultdict

In [ ]:
mutation_rate = 

In [160]:
# Load the tree sequences

filename = "chr12_HapMapII_GRCh37_CEU500_seed1"
percent = "10"

# reference ts from stdpopsim
ts_std = tskit.load(f"../../stdpopsim_simulations/simulations/{filename}.trees")
# masked ts from Relate
ts_masked = tskit.load(f"../../Relate_sims/masked_{filename}_{percent}.00percent_25bins_seed42.trees")

# load in hidden SNP indices
hidden_snps_df = pd.read_csv(f"../../Relate_sims/hidden_snps_key_{filename}_{percent}.00percent_25bins_seed42.csv")

# load in the full .hapsfile
path = f"../Relate_sims/data/{filename}.haps"

# format the haps file into a pandas dataframe
haps_df = pd.read_csv(f"../../Relate_sims/data/{filename}.haps", sep=r"\s+", header=None)
# format the haps file into a pandas dataframe
masked_haps_df = pd.read_csv(f"../../Relate_sims/data/masked_{filename}_{percent}.00percent_25bins_seed42.haps", sep=r"\s+", header=None)

haps_df = haps_df.rename(columns={
    0: "Chromosome number",
    1: "SNP ID",
    2: "SNP position",
    3: "Ancestral allele",
    4: "Alternative allele",
})

masked_haps_df = masked_haps_df.rename(columns={
    0: "Chromosome number",
    1: "SNP ID",
    2: "SNP position",
    3: "Ancestral allele",
    4: "Alternative allele",
})


In [161]:
masked_haps_df

,Chromosome number,SNP ID,SNP position,Ancestral allele,Alternative allele,5,6,7,8,9,...,997,998,999,1000,1001,1002,1003,1004,1005,1006
0,12,0,148699,C,G,0,0,0,0,0,...,0,0,0,0,0,0,0,0,3,0
1,12,1,149430,T,G,0,0,0,0,0,...,0,0,0,0,0,0,0,0,8,0
2,12,3,149553,T,A,0,0,0,0,0,...,0,0,0,0,0,0,0,0,1,0
3,12,4,149840,G,A,0,0,0,1,0,...,0,0,0,0,0,0,0,0,1,0
4,12,5,149999,G,T,0,0,0,0,0,...,0,0,0,0,0,0,0,0,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
438775,12,488152,133273381,T,C,0,0,0,0,0,...,0,0,0,0,0,0,0,0,2,0
438776,12,488153,133273564,T,C,0,0,0,0,0,...,0,0,0,0,0,0,0,0,4,0
438777,12,488154,133274494,T,A,0,0,0,0,0,...,0,0,0,0,0,0,0,0,3,0
438778,12,488155,133274684,G,A,0,0,0,0,0,...,0,0,0,0,0,0,0,0,10,0


In [162]:
haps_df

,Chromosome number,SNP ID,SNP position,Ancestral allele,Alternative allele,5,6,7,8,9,...,995,996,997,998,999,1000,1001,1002,1003,1004
0,12,0,148699,C,G,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,12,1,149430,T,G,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,12,2,149532,C,T,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,12,3,149553,T,A,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,12,4,149840,G,A,0,0,0,1,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
487528,12,488154,133274494,T,A,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
487529,12,488155,133274684,G,A,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
487530,12,488156,133274833,A,T,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
487531,12,488157,133274967,G,T,0,0,0,1,1,...,0,0,1,1,0,0,1,0,1,1


In [254]:
hidden_snps_df.head(20)

,Unnamed: 0,Chromosome number,SNP ID,SNP position,Ancestral allele,Alternative allele,5,6,7,8,...,997,998,999,1000,1001,1002,1003,1004,ALT_count,ALT_count_numbered_bin
0,478742,12,479355,130919180,A,T,0,0,0,0,...,0,0,0,0,0,0,0,0,1,0
1,330678,12,331094,91013145,C,T,0,0,0,0,...,0,0,0,0,0,0,0,0,1,0
2,85786,12,85891,24190668,A,C,0,0,0,0,...,0,0,0,0,0,0,0,0,2,0
3,367077,12,367544,100836424,G,T,0,0,0,0,...,0,0,0,0,0,0,0,0,1,0
4,97645,12,97766,27350327,A,C,0,0,0,0,...,0,0,0,0,0,0,0,0,10,0
5,297211,12,297590,81694959,T,G,0,0,0,0,...,0,0,0,0,0,0,0,0,4,0
6,183904,12,184137,50685621,T,A,0,0,0,0,...,0,0,1,0,0,0,0,0,119,2
7,374298,12,374768,102790020,C,T,0,0,0,0,...,0,0,0,0,0,0,0,0,1,0
8,347498,12,347933,95676605,C,A,0,0,0,0,...,0,0,0,0,0,0,0,0,13,0
9,13398,12,13416,3843630,C,A,0,0,0,0,...,0,0,0,0,0,0,0,0,1,0


## single tree example

In [255]:
pos = 81694959	

In [256]:
def snps_by_tree_index(ts, snp_positions):
    positions = np.asarray(snp_positions)

    breakpoints = ts.breakpoints(as_array=True)
    tree_indices = np.searchsorted(breakpoints, positions, side="right") - 1

    # skip SNPs outside the sequence, or on a tree with no edges (e.g. before Relate's first tree, in masked relate regions, or snps that are left out at the end of the chromosome)
    inside = (positions >= 0) & (positions < ts.sequence_length)
    empty = np.array([tree.num_edges == 0 for tree in ts.trees()])
    keep = inside.copy()
    keep[inside] = ~empty[tree_indices[inside]]
    if not keep.all():
        print(f"skipped {(~keep).sum()} SNPs outside the sequence or on empty trees")

    result = defaultdict(list)
    for tree_index, pos in zip(tree_indices[keep].tolist(), positions[keep].tolist()):
        result[tree_index].append(pos)

    return dict(result)

In [257]:
def haplotype_columns(df):
    """Columns named with a number (5, 6, ..., or "5", "6", ...), in numeric order."""
    cols = [c for c in df.columns if str(c).isdigit()]
    return sorted(cols, key=int)


In [258]:
def tip_states_at(df, pos):
    """0/1 tip states for the SNP at `pos`, as {sample 0..n-1: state}."""
    hap_cols = haplotype_columns(df)
    row = df.loc[df["SNP position"] == pos, hap_cols]
    if len(row) != 1:
        raise ValueError(f"{len(row)} rows at position {pos}")
    haps = row.to_numpy().ravel().astype(int)
    return dict(enumerate(haps))      # {0: state, 1: state, ...}

In [259]:
def mutation_rate_from_provenance(ts):
    """The rate passed to msprime.sim_mutations, read from the provenance records."""
    rates = []
    for p in ts.provenances():
        record = json.loads(p.record)
        params = record.get("parameters", {})
        if params.get("command") == "sim_mutations":
            rates.append(params.get("rate"))
    return rates

In [260]:
snp_to_tree_std = snps_by_tree_index(ts_std, hidden_snps_df["SNP position"])
snp_to_tree_masked= snps_by_tree_index(ts_masked, hidden_snps_df["SNP position"])

skipped 2 SNPs outside the sequence or on empty trees


In [261]:
tips = tip_states_at(hidden_snps_df, pos)

In [262]:
hap_cols = haplotype_columns(hidden_snps_df)
biallelic = hidden_snps_df[hidden_snps_df[hap_cols].max(axis=1) <= 1]["SNP position"]

in_std = {p for ps in snp_to_tree_std.values() for p in ps}
in_masked = {p for ps in snp_to_tree_masked.values() for p in ps}

usable = [p for p in biallelic if p in in_std and p in in_masked]
print(len(hidden_snps_df), "hidden SNPs,", len(usable), "usable")

48753 hidden SNPs, 48751 usable


In [263]:
lengths = ts_masked.nodes_time[ts_masked.edges_parent] - ts_masked.nodes_time[ts_masked.edges_child]
print("zero-length branches:", (lengths <= 0).sum())

zero-length branches: 0


In [264]:
ts_std

In [277]:
tips = tip_states_at(hidden_snps_df, pos)
rate_std = mutation_rate_from_provenance(ts_std)[0]
rate_relate = 1.25e-8


In [278]:

out_std = felsenstein_backwards_general(ts_std.at(pos), tips, bases=(0, 1),
                                        mutation_rate=rate_std, root_state=0)
out_relate = felsenstein_backwards_general(ts_masked.at(pos), tips, bases=(0, 1),
                                           mutation_rate=rate_relate, root_state=0)


In [279]:
_, states, path_probability, num_mutations = out_relate

In [280]:
print("path probability:", path_probability)
print("mutations:", num_mutations)

path probability: 4.246039842249721e-07
mutations: 1


In [281]:
# where are the mutations on this tree
tree = ts_masked.at(pos)
for u in tree.nodes():
    p = tree.parent(u)
    if p != -1 and states[u] != states[p]:
        print(f"mutation on branch above node {u}: {states[p]} -> {states[u]}, "
              f"{tree.num_samples(u)} samples below, branch length {tree.branch_length(u):.0f}")

mutation on branch above node 9926740: 0 -> 1, 4 samples below, branch length 34


In [282]:
# true mutations !!!
site = ts_std.site(position=pos)
print("true mutations:", len(site.mutations))
for m in site.mutations:
    tree = ts_std.at(pos)
    print(f"  above node {m.node} ({tree.num_samples(m.node)} samples below), "
          f"derived state {m.derived_state}")
    

true mutations: 1
  above node 24408 (4 samples below), derived state G


In [283]:
def mutation_branches(tree, states):
    """Branches where the reconstruction changes state: (child node, parent node)."""
    return [(u, tree.parent(u)) for u in states
            if tree.parent(u) != -1 and states[u] != states[tree.parent(u)]]


def compare_snp(pos, tips, ts_std, ts_masked, rate_std, rate_relate):
    """One row of results for one hidden SNP."""
    row = {"position": pos, "carriers": int(sum(tips.values()))}

    # the truth: what stdpopsim actually simulated
    site = ts_std.site(position=pos)
    row["true_mutations"] = len(site.mutations)
    row["true_mutation_time"] = site.mutations[0].time if len(site.mutations) == 1 else np.nan

    for name, ts, rate in (("std", ts_std, rate_std), ("relate", ts_masked, rate_relate)):
        tree = ts.at(pos)
        likelihoods, states, path_probability, num_mutations = felsenstein_backwards_general(
            tree, tips, bases=(0, 1), mutation_rate=rate, root_state=0)
        row[f"logL_{name}"] = np.log(likelihoods[tree.root][0])   # all reconstructions, root = 0
        row[f"log_path_{name}"] = np.log(path_probability)        # best reconstruction only
        row[f"mutations_{name}"] = num_mutations

        # when: the time span of the branch carrying the mutation (if there is just one)
        branches = mutation_branches(tree, states)
        if len(branches) == 1:
            u, p = branches[0]
            row[f"branch_low_{name}"] = ts.nodes_time[u]
            row[f"branch_high_{name}"] = ts.nodes_time[p]
        else:
            row[f"branch_low_{name}"] = row[f"branch_high_{name}"] = np.nan
    return row

In [284]:
rows = [compare_snp(pos, tip_states_at(hidden_snps_df, pos), ts_std, ts_masked, rate_std, rate_relate)
        for pos in usable[:200]]
results = pd.DataFrame(rows)

In [285]:
print("total log-likelihood   true:", results.logL_std.sum(), "  relate:", results.logL_relate.sum())
print("SNPs with 1 mutation   true:", (results.mutations_std == 1).mean(),
      "  relate:", (results.mutations_relate == 1).mean())

one = results.dropna(subset=["true_mutation_time", "branch_low_relate"])
inside = (one.true_mutation_time >= one.branch_low_relate) & (one.true_mutation_time <= one.branch_high_relate)
print("true mutation time inside Relate's branch:", inside.mean())

total log-likelihood   true: -2328.5597787497736   relate: -4297.097205800426
SNPs with 1 mutation   true: 1.0   relate: 0.75
true mutation time inside Relate's branch: 0.5666666666666667


# All valid sites run

In [288]:
# mutation rates
rate_std = mutation_rate_from_provenance(ts_std)[0]   # true rate from stdpopsim
rate_relate = 1.25e-8                                  # the -m you gave Relate
print("rates:", rate_std, rate_relate)

# usable SNPs: two alleles only, and a tree in both tree sequences
hap_cols = haplotype_columns(hidden_snps_df)
assert len(hap_cols) == ts_std.num_samples == ts_masked.num_samples

biallelic = hidden_snps_df[hidden_snps_df[hap_cols].max(axis=1) <= 1]["SNP position"]
in_std = {p for ps in snp_to_tree_std.values() for p in ps}
in_masked = {p for ps in snp_to_tree_masked.values() for p in ps}
usable = [p for p in biallelic if p in in_std and p in in_masked]
print(len(hidden_snps_df), "hidden SNPs,", len(usable), "usable")

rates: 1.29e-08 1.25e-08
48753 hidden SNPs, 48751 usable
